# Milestone 2 — Ravi Varma style LoRA

Trains the LoRA from `scripts/train_lora.py`. The logic lives in the repo, not in
this notebook, so a run is reproducible and reviewable — the notebook only wires up
the GPU, Drive and the dataset.

**Before the first run**, put the training split in Drive at
`My Drive/ravi-varma/train/` — the 100 images and `metadata.jsonl`, 126 MB. Drag the
`data/train` folder into drive.google.com, or use `rclone`/`gdown` if you prefer a
CLI. It only has to happen once.

The trainer reads each image exactly once, to build the latent cache, and never
touches the files again — so the Drive mount is fast enough to train directly off,
and there is no copy to keep in sync.

**Runtime → Change runtime type → T4 GPU** before running anything below.


## 1. Check the GPU


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'
print(torch.__version__, torch.cuda.get_device_name(0))


## 2. Dependencies

`torch` is left alone on purpose — Colab's build matches its CUDA driver and
replacing it is the usual way a session breaks.


In [ ]:
!git clone -q https://github.com/ria842/next-ravi-varma.git /content/repo || (cd /content/repo && git pull -q)
%cd /content/repo
!pip install -q -r notebooks/requirements-gpu.txt
import diffusers, peft, transformers
print('diffusers', diffusers.__version__, '| peft', peft.__version__, '| transformers', transformers.__version__)


## 3. Mount Drive

Checkpoints go **to Drive**, not to `/content` — Colab will kill this session and
`/content` dies with it.

The images are read straight off the mount. Drive's per-file latency is real but it
is paid once: `train_lora.py` decodes every image a single time during the latent
cache, then trains on the cached tensors. Nothing is copied, unzipped or duplicated,
so there is no second copy that can drift from `data/train`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, pathlib

MYDRIVE = '/content/drive/MyDrive'
RUN     = 'r32-512-bucket'      # change per run; each gets its own Drive folder

# Find the split wherever it landed. Drag-and-drop puts it under data/train or
# train depending on what you dragged, and the project folder gets misspelled
# ravi-verma about half the time -- so locate metadata.jsonl instead of guessing.
# Check the likely paths first: a recursive walk of a large Drive over FUSE is
# slow, so it is the fallback, not the first move.
CANDIDATES = [f'{MYDRIVE}/{d}/{sub}'
              for d in ('ravi-varma', 'ravi-verma', 'next-ravi-varma', '')
              for sub in ('data/train', 'train')]
hits = [f'{d}/metadata.jsonl' for d in CANDIDATES
        if os.path.exists(f'{d}/metadata.jsonl')]
if not hits:
    print('not in the usual places; walking Drive, this can take a minute...')
    hits = sorted(glob.glob(f'{MYDRIVE}/**/metadata.jsonl', recursive=True))
hits = [h for h in hits if 'holdout' not in h]
if not hits:
    raise SystemExit(
        f'No metadata.jsonl found anywhere under {MYDRIVE}.\n'
        'Upload the data/train folder (100 images + metadata.jsonl) to Drive,'
        ' then re-run this cell.')
if len(hits) > 1:
    print('Several candidates found; using the first. Set DATA by hand to override:')
    for h in hits:
        print('   ', os.path.dirname(h))

DATA  = os.path.dirname(hits[0])
DRIVE = os.path.dirname(DATA)
if os.path.basename(DRIVE) == 'data':      # .../ravi-varma/data/train
    DRIVE = os.path.dirname(DRIVE)

n_images = len([f for f in os.listdir(DATA) if f.lower().endswith(('.jpg', '.png'))])
n_rows   = sum(1 for line in open(f'{DATA}/metadata.jsonl') if line.strip())
print(f'DATA  = {DATA}')
print(f'DRIVE = {DRIVE}   (checkpoints -> {DRIVE}/runs/{RUN})')
print(f'{n_images} images, {n_rows} caption rows')
assert n_images == n_rows == 100, (
    f'expected 100 images and 100 rows, got {n_images} and {n_rows} -- '
    'the upload is incomplete, let it finish before training')

pathlib.Path(f'{DRIVE}/runs/{RUN}').mkdir(parents=True, exist_ok=True)

# If the mount turns out to be slow or flaky mid-cache, copy to local disk
# first -- 126 MB, under a minute:  !cp -r '{DATA}' /content/train


## 4. Train

Hyperparameters come from `plans/PLAN.md` §3 Milestone 2 and are the script's
defaults: rank 32, alpha 32, lr 1e-4 constant-with-warmup, effective batch 4 via
gradient accumulation, fp16, 8-bit Adam, gradient checkpointing, text encoder
frozen, 2500 steps.

Two things worth knowing before you read the output:

- **Resolution is 512, not 768.** 49 of the 100 images have a short edge under 512
  (median 528), so 768 would upscale most of the corpus and teach the LoRA
  interpolation artefacts as if they were brushwork.
- **Aspect-ratio bucketing is on.** 93 of 100 paintings are portrait, median ratio
  0.70. A square crop keeps a median 70% of each painting; bucketing keeps 92% and
  upscales nothing. It also matters that our captions carry a *composition* field —
  cropping would make that field describe a framing the model never sees.

If the session dies, re-run cells 1–3 and then this cell with `--resume latest`.


In [ ]:
!python scripts/train_lora.py \
    --data-dir '{DATA}' \
    --output-dir '{DRIVE}/runs/{RUN}' \
    --rank 32 \
    --resolution 512 \
    --max-train-steps 2500 \
    --checkpointing-steps 500 \
    --validation-steps 500


### Resume after a killed session

Checkpoints carry the optimiser moments as well as the adapter weights, so a resume
continues the run rather than restarting the schedule.


In [ ]:
# !python scripts/train_lora.py \
#     --data-dir '{DATA}' \
#     --output-dir '{DRIVE}/runs/{RUN}' \
#     --resume latest


## 5. The loss curve

`loss.csv` is appended and flushed every logging step, so it survives a kill. A
resumed run appends to the same file — the step column is what to plot against, not
row order.


In [ ]:
import pandas as pd, matplotlib.pyplot as plt

df = pd.read_csv(f'{DRIVE}/runs/{RUN}/loss.csv').sort_values('step')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(df.step, df.loss, lw=0.8, alpha=0.5, label='logged')
ax[0].plot(df.step, df.loss.rolling(10, min_periods=1).mean(), lw=2, label='10-pt mean')
ax[0].set_xlabel('step'); ax[0].set_ylabel('MSE loss'); ax[0].legend()
ax[0].set_title(f'{RUN} — training loss')
ax[1].plot(df.step, df.lr); ax[1].set_xlabel('step'); ax[1].set_ylabel('lr')
ax[1].set_title('learning rate')
plt.tight_layout()
plt.savefig(f'{DRIVE}/runs/{RUN}/loss_curve.png', dpi=150)
plt.show()

print(df.tail())
# diffusion loss is dominated by which timestep got sampled, so it is noisy and
# nearly flat. It tells you the run did not diverge; it does NOT tell you the
# style was learned. That is what Milestone 4's evaluation is for.



## 6. Does the trigger actually do anything?

The top entry in the §5 risk register is *trigger token does nothing — style leaked
into caption words*. This is the test, and it should be run on the first checkpoint,
not at the end of the week.

Three comparisons, same seed throughout:

1. base SD 1.5, no LoRA — the control
2. LoRA loaded, prompt **without** `rvrma`
3. LoRA loaded, prompt **with** `rvrma`

If 2 and 3 look the same, the trigger is inert and the style is riding on the LoRA
weight alone. That is a publishable finding for the report, not a failure — but you
need to know which it is.


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

BASE = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
# whatever has been written so far -- run this at step 500, do not wait for 2500
import glob
CKPT = sorted(glob.glob(f'{DRIVE}/runs/{RUN}/checkpoint-*'))[-1]
print('testing', CKPT)

pipe = StableDiffusionPipeline.from_pretrained(BASE, torch_dtype=torch.float16,
                                               safety_checker=None,
                                               requires_safety_checker=False).to('cuda')
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

SCENE = 'Shakuntala writing a letter, seated under a tree'
SEED  = 42

def gen(prompt, scale=None):
    g = torch.Generator('cuda').manual_seed(SEED)
    kw = {'cross_attention_kwargs': {'scale': scale}} if scale is not None else {}
    return pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=g, **kw).images[0]

panel = [('base SD 1.5 (no LoRA)', gen(SCENE))]
pipe.load_lora_weights(CKPT)
panel.append(('LoRA, no trigger', gen(SCENE)))
panel.append(('LoRA + rvrma', gen(f'rvrma, {SCENE}')))

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
for ax, (title, img) in zip(axes, panel):
    ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{DRIVE}/runs/{RUN}/trigger_test.png', dpi=130); plt.show()


## 7. LoRA weight sweep

The adapter scale is the real strength control. Somewhere past ~1.0 the style starts
eating the content; find that point now so Milestone 3 can ship a sane default.


In [ ]:
scales = [0.0, 0.4, 0.7, 1.0, 1.3]
imgs = [(f'scale {s}', gen(f'rvrma, {SCENE}', scale=s)) for s in scales]

fig, axes = plt.subplots(1, len(imgs), figsize=(4*len(imgs), 4.5))
for ax, (title, img) in zip(axes, imgs):
    ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{DRIVE}/runs/{RUN}/scale_sweep.png', dpi=130); plt.show()


## 8. Planned runs

Change `RUN` in cell 3 and re-run. Each lands in its own Drive folder with its own
`loss.csv`, so the curves can be overlaid in the report.

| `RUN` | flags | question |
|---|---|---|
| `r32-512-bucket` | defaults | the reference run |
| `r16-512-bucket` | `--rank 16` | does style need rank 32? |
| `r32-512-square` | `--no-bucket` | was bucketing worth it? |
| `r32-512-photoreal` | `--base-model emilianJR/epiCRealism` | §6 open question |

The text-encoder-trained variant in §3 is deliberately not here — `train_lora.py`
freezes the text encoder by design, and unfreezing it is a separate change to make
once the frozen baselines are in hand.

A photoreal checkpoint must be in **diffusers** layout with a `vae/` subfolder —
the `*_noVAE` Realistic Vision repos are not, and a single-file `.safetensors`
needs `from_single_file` conversion first.
